# Steam indie games: data preparation

Load `games.csv`, check source coverage, and prepare the Indie-labelled sample released from January 1, 2019 through December 25, 2025. Reception comparisons use at least 100 recorded reviews.

This notebook contains preparation and source audits. The genre and achievement analyses are in separate notebooks.


In [1]:
%matplotlib inline

import csv

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)

## 1. Load and validate the data

The CSV header contains 39 labels, while its records contain 40 fields.
The header combines `Discount` and `DLC count` into `DiscountDLC count`.

Loading the file without correcting this header misaligns the columns.
I will separate these labels during import and preserve the original file.

In [2]:
with open("games.csv", encoding="utf-8-sig", newline="") as file:
    reader = csv.reader(file)

    original_columns = next(reader)
    first_record = next(reader)

print("Header labels:", len(original_columns))
print("Values in first record:", len(first_record))

Header labels: 39
Values in first record: 40


In [3]:
columns = original_columns.copy()

position = columns.index("DiscountDLC count")

columns[position:position + 1] = ["Discount", "DLC count"]

print("Number of labels:", len(columns))

Number of labels: 40


In [4]:
df = pd.read_csv(
    "games.csv",
    header=0,
    names=columns,
    low_memory=False,
)

df[["AppID", "Name", "Price", "Discount", "DLC count"]].head()

,AppID,Name,Price,Discount,DLC count
0,2539430,Black Dragon Mage Playtest,0.00,0,0
1,496350,Supipara - Chapter 1 Spring Has Come!,5.24,65,0
2,1034400,Mystery Solitaire The Black Raven,4.99,0,0
3,3292190,버튜버 파라노이아 - Vtuber Paranoia,8.99,0,1
4,3631080,Maze Quest VR,4.99,0,0


In [5]:
print("Rows and columns:", df.shape)

df.info()

Rows and columns: (125855, 40)
<class 'pandas.DataFrame'>
RangeIndex: 125855 entries, 0 to 125854
Data columns (total 40 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125855 non-null  int64  
 1   Name                        125854 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  str    
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  int64  
 6   Price                       125855 non-null  float64
 7   Discount                    125855 non-null  int64  
 8   DLC count                   125855 non-null  int64  
 9   About the game              117393 non-null  str    
 10  Supported languages         125855 non-null  str    
 11  Full audio languages        125855 non-null  str    
 12  Reviews                     12181 non-null   str    

In [6]:
missing = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_pct": df.isna().mean() * 100,
})

missing.sort_values(
    "missing_count",
    ascending=False,
).round(2)

,missing_count,missing_pct
Movies,125855,100.00
Score rank,125815,99.97
Metacritic url,121597,96.62
Reviews,113674,90.32
Notes,102545,81.48
Website,75335,59.86
Support url,70560,56.06
Tags,42502,33.77
Support email,22630,17.98
Categories,8966,7.12


Exclude media and contact fields because they do not support the current questions. Exclude Score rank because only 40 entries have values. Preserve the original DataFrame.

In [7]:
columns_to_exclude = [
    "Movies",
    "Score rank",
    "Header image",
    "Screenshots",
    "Website",
    "Support url",
    "Support email",
    "Metacritic url",
]

analysis_df = df.drop(columns=columns_to_exclude).copy()

In [8]:
checks = pd.Series({
    "Missing AppIDs": analysis_df["AppID"].isna().sum(),
    "Duplicate AppIDs": analysis_df["AppID"].duplicated().sum(),
    "Missing names": analysis_df["Name"].isna().sum(),
    "Negative prices": (analysis_df["Price"] < 0).sum(),
    "Negative positive-review counts": (analysis_df["Positive"] < 0).sum(),
    "Negative negative-review counts": (analysis_df["Negative"] < 0).sum(),
    "Discounts outside 0–100": (
        ~analysis_df["Discount"].between(0, 100)
    ).sum(),
})

checks

Missing AppIDs                     0
Duplicate AppIDs                   0
Missing names                      1
Negative prices                    0
Negative positive-review counts    0
Negative negative-review counts    0
Discounts outside 0–100            0
dtype: int64

In [9]:
analysis_df.loc[
    analysis_df["Name"].isna(),
    ["AppID", "Name", "Price", "Positive", "Negative", "Genres"],
]

,AppID,Name,Price,Positive,Negative,Genres
44432,396420,NaN,0.0,0,0,NaN


One source entry has a missing name and remains in the broader catalog, identified by AppID. It has zero recorded reviews and zero recorded price without a Free To Play label, so it is excluded from the final pricing population under the zero-price rule.

## Review metrics

`total_reviews` is the sum of positive and negative reviews. `positive_share` is the positive fraction, from 0 to 1. Entries without reviews have undefined reception, represented by NaN. Review-volume counts below describe the population before date and product exclusions.

In [10]:
analysis_df["total_reviews"] = (
    analysis_df["Positive"] + analysis_df["Negative"]
)

analysis_df["positive_share"] = (
    analysis_df["Positive"]
    / analysis_df["total_reviews"].replace(0, np.nan)
)

analysis_df[
    ["Name", "Positive", "Negative", "total_reviews", "positive_share"]
].head(10)

,Name,Positive,Negative,total_reviews,positive_share
0,Black Dragon Mage Playtest,0,0,0,NaN
1,Supipara - Chapter 1 Spring Has Come!,252,3,255,0.988235
2,Mystery Solitaire The Black Raven,21,3,24,0.875000
3,버튜버 파라노이아 - Vtuber Paranoia,0,0,0,NaN
4,Maze Quest VR,0,0,0,NaN
5,Agony VR,0,0,0,NaN
6,Armored Brigade II,117,13,130,0.900000
7,Galacatraz: Eject Equip Escape,0,0,0,NaN
8,Hepta Beats,28,4,32,0.875000
9,MUMBA IV: Egypt Jewels,19,6,25,0.760000


In [11]:
pd.Series({
    "No reviews": (analysis_df["total_reviews"] == 0).sum(),
    "1–9 reviews": analysis_df["total_reviews"].between(1, 9).sum(),
    "10–99 reviews": analysis_df["total_reviews"].between(10, 99).sum(),
    "100+ reviews": (analysis_df["total_reviews"] >= 100).sum(),
})

No reviews       42899
1–9 reviews      26294
10–99 reviews    33858
100+ reviews     22804
dtype: int64

In [12]:
analysis_df["release_date"] = pd.to_datetime(
    analysis_df["Release date"],
    format="mixed",
    errors="coerce",
)

pd.Series({
    "Unparsed dates": analysis_df["release_date"].isna().sum(),
    "Earliest date": analysis_df["release_date"].min(),
    "Latest date": analysis_df["release_date"].max(),
})

Unparsed dates                      0
Earliest date     1997-06-30 00:00:00
Latest date       2026-12-01 00:00:00
dtype: object

## 2. Audit review coverage and define the historical scope

The Kaggle update date does not establish when each field or entry was collected. Release dates describe titles, not observation dates. The earlier October 2026 cutoff is replaced by a documented historical scope below.

Zero positive and negative review counts are recorded zeros, not null values. They may reflect genuinely absent reviews or incomplete collection; this file cannot distinguish the causes. Other fields in those rows may still contain useful information.

In [13]:
# Audit the complete source population before product/date exclusions.
review_coverage_by_year = (
    analysis_df.assign(release_year=analysis_df["release_date"].dt.year)
    .groupby("release_year")
    .agg(
        entries=("AppID", "size"),
        entries_with_reviews=("total_reviews", lambda x: x.gt(0).sum()),
        entries_with_100_reviews=("total_reviews", lambda x: x.ge(100).sum()),
    )
)
review_coverage_by_year["pct_with_reviews"] = (
    review_coverage_by_year["entries_with_reviews"]
    / review_coverage_by_year["entries"] * 100
)
latest_release_with_reviews = analysis_df.loc[
    analysis_df["total_reviews"].gt(0), "release_date"
].max()
print("Latest recorded release date with nonzero reviews:",
      latest_release_with_reviews.date())
review_coverage_by_year.tail(10).round(1)


Latest recorded release date with nonzero reviews: 2025-12-25


,entries,entries_with_reviews,entries_with_100_reviews,pct_with_reviews
release_year,,,,
2017,5922,5823,1927,98.3
2018,7461,7184,1902,96.3
2019,7243,6674,1765,92.1
2020,8804,8021,2000,91.1
2021,11067,8537,1867,77.1
2022,12288,8665,1749,70.5
2023,14598,9700,1927,66.4
2024,20033,12439,2280,62.1
2025,24973,5872,1102,23.5


In [14]:
# Check specific titles by AppID, so naming differences do not hide them.
fc_ids = {3405690: "FC 26", 4080220: "FC 27"}
fc_source_check = (
    analysis_df.set_index("AppID")
    .reindex(list(fc_ids))[
        ["Name", "Release date", "Price", "Discount", "Positive", "Negative"]
    ]
)
fc_source_check.insert(0, "Expected title", list(fc_ids.values()))
fc_source_check["Present in source"] = fc_source_check.index.isin(analysis_df["AppID"])
fc_source_check

,Expected title,Name,Release date,Price,Discount,Positive,Negative,Present in source
AppID,,,,,,,,
3405690,FC 26,EA SPORTS FC™ 26,"Sep 25, 2025",27.99,60.0,0.0,0.0,True
4080220,FC 27,NaN,NaN,NaN,NaN,NaN,NaN,False


### Coverage finding and interpretation

In this file, all 3,316 entries dated 2026 have zero recorded positive and negative reviews. The latest recorded release date with any nonzero reviews is December 25, 2025. This is an observed coverage boundary, not a verified collection date or proof that every earlier record is complete.

FC 26 is present under AppID 3405690 with a positive recorded price and zero recorded reviews. FC 27's checked AppID is absent. The earlier claim that both titles were absent was incorrect; matching by AppID resolves the naming ambiguity. FC 26 shows why an earlier release-date cutoff alone cannot repair incomplete review coverage.

Retain the source data and the product-filtered catalog. Create a separate historical population for the pricing analysis; do not convert recorded zeros to invented reviews or prices.

In [15]:
source_genre_counts = (
    analysis_df["Genres"]
    .str.split(",")
    .explode()
    .str.strip()
    .value_counts()
)

source_genre_counts

Genres
Indie                    82884
Casual                   51677
Action                   47344
Adventure                46293
Simulation               24912
Strategy                 23058
RPG                      21577
Free To Play             12605
Early Access             11572
Sports                    4971
Racing                    4169
Massively Multiplayer     2596
Utilities                 1091
Design & Illustration      584
Animation & Modeling       484
Education                  464
Violent                    414
Video Production           340
Game Development           323
Gore                       257
Audio Production           229
Software Training          222
Photo Editing              183
Web Publishing             113
Nudity                      94
Sexual Content              88
Accounting                  32
Movie                        2
Documentary                  1
Episodic                     1
Short                        1
Tutorial                     1
3

We exclude entries carrying the listed software or video genre labels
to focus the analysis on games. This is a label-based rule and may
exclude hybrid products. Entries with missing genres remain unclassified.

In [16]:
non_game_labels = {
    "Utilities",
    "Design & Illustration",
    "Animation & Modeling",
    "Video Production",
    "Game Development",
    "Audio Production",
    "Software Training",
    "Photo Editing",
    "Web Publishing",
    "Accounting",
    "Movie",
    "Documentary",
    "Episodic",
    "Short",
    "Tutorial",
    "360 Video",
}

genre_lists = (
    analysis_df["Genres"]
    .fillna("")
    .str.split(",")
    .apply(lambda labels: {label.strip() for label in labels})
)

possible_non_game = genre_lists.apply(
    lambda labels: bool(labels & non_game_labels)
)


analysis_df.loc[
    possible_non_game,
    ["Name", "Genres", "Categories", "Price", "total_reviews"],
].head(20)

,Name,Genres,Categories,Price,total_reviews
31,DrumKit VR - Play drum kit in the world of VR,Audio Production,"Tracked Controller Support,VR Only",4.99,22
82,VSS - Virtual Streaming Software,"Animation & Modeling,Video Production,Early Ac...",NaN,0.00,0
182,SpriteStack,"Animation & Modeling,Design & Illustration,Gam...",NaN,29.99,95
210,Desktop Companion,"Animation & Modeling,Design & Illustration,Uti...","Steam Achievements,Steam Workshop",1.24,0
292,Félix VR,Video Production,"Tracked Controller Support,VR Only,Partial Con...",0.00,22
315,Gaming Copilot: AI Companion,"Indie,Utilities","Single-player,Steam Achievements,Full controll...",4.39,80
498,Pincel de Fractales,"Casual,Indie,Design & Illustration","Mouse Only Option,Playable without Timed Input",1.99,4
532,Live Wallpaper,"Casual,Free To Play,Animation & Modeling,Desig...",Steam Workshop,0.00,0
544,The Bond,Animation & Modeling,"Tracked Controller Support,VR Supported",0.00,163
555,Mind Massaging Machine,Utilities,"Tracked Controller Support,VR Supported,Partia...",19.99,16


In [17]:
print("Before exclusion:", len(analysis_df))
print("Software/video entries excluded:", possible_non_game.sum())

analysis_df = analysis_df.loc[~possible_non_game].copy()

print("After exclusion:", len(analysis_df))

Before exclusion: 125855
Software/video entries excluded: 1726
After exclusion: 124129


## 3. Historical pricing population

Restrict recorded release dates to December 25, 2025 or earlier, the latest release date with nonzero recorded reviews. Entries after this boundary have recorded zeros for reviews, not entirely null records. This is a scope restriction; it does not repair incomplete fields in retained entries or establish the collection date.

`analysis_df` preserves the product-filtered catalog. `historical_df` additionally excludes explicit Playtest/Demo title matches and supports all subsequent pricing tables and threshold comparisons. `indie_focus_reviewed_df` further requires 100 recorded reviews after the Indie/date restriction. Prices are recorded USD values, not verified current prices or launch prices.

For pricing comparisons, retain positive recorded prices and zero recorded prices only when the entry also carries the exact Free To Play label. Preserve rejected zero-price entries in `excluded_zero_price_entries`. The retained zero-price group is labelled `0 (F2P label)` and is a separate descriptive reference, not independently verified free status. This rule applies before review thresholds and before genre expansion.

### 3.1 Apply the historical release-date cutoff

In [18]:
review_date_cutoff = pd.Timestamp("2025-12-25")
after_review_cutoff = analysis_df["release_date"].gt(review_date_cutoff)

historical_candidates = analysis_df.loc[
    analysis_df["release_date"].le(review_date_cutoff)
].copy()

### 3.2 Exclude Playtest/Demo title matches

Exclude entries whose titles contain the separate English words Playtest or Demo, case-insensitively, before calculating pricing summaries. Preserve matches in `excluded_test_demo_entries` for inspection. This heuristic may miss localized labels or unusual title formats; it is not a complete product-type classification. Prologues remain included because the title alone does not establish that they are demos.

All downstream pricing, review-threshold, and genre results use the retained historical population. The original source and product-filtered catalog remain available.

In [19]:
# Explicit English title markers only; prologues remain in scope.
test_or_demo = historical_candidates["Name"].str.contains(
    r"\b(?:playtest|demo)\b", case=False, na=False
)
excluded_test_demo_entries = historical_candidates.loc[
    test_or_demo, ["AppID", "Name", "Price", "total_reviews"]
].copy()
historical_before_price_rule = historical_candidates.loc[~test_or_demo].copy()

In [20]:
excluded_test_demo_entries.head(20)

,AppID,Name,Price,total_reviews
0,2539430,Black Dragon Mage Playtest,0.0,0
10,1946890,Codename: Warlock Playtest,0.0,0
24,2349750,CyberVault Playtest,0.0,0
25,2628280,A Night In Omar's Burger Playtest,0.0,0
51,2689730,Dark and Deep Playtest,0.0,0
97,2782210,Bill's Misfortune Playtest,0.0,0
102,3664570,QUADRA CLASH Playtest,0.0,0
116,3581240,Descendance Playtest,0.0,0
118,1549730,Red Solstice 2: Survivors Playtest,0.0,0
137,1639330,Pain Party Playtest,0.0,0


### 3.3 Apply the zero-price eligibility rule

Keep positive recorded prices. Keep zero recorded prices only with the exact Free To Play label. Preserve all other zero-price entries for auditing; do not alter their source values.

In [21]:
# Exact membership in the comma-separated genre labels, not a substring match.
has_free_to_play_label = (
    historical_before_price_rule["Genres"]
    .fillna("")
    .str.split(",")
    .apply(lambda labels: "Free To Play" in {label.strip() for label in labels})
)
ambiguous_zero_price = (
    historical_before_price_rule["Price"].eq(0)
    & ~has_free_to_play_label
)
excluded_zero_price_entries = historical_before_price_rule.loc[
    ambiguous_zero_price
].copy()

pricing_eligible = (
    historical_before_price_rule["Price"].gt(0)
    | (historical_before_price_rule["Price"].eq(0) & has_free_to_play_label)
)
historical_df = historical_before_price_rule.loc[pricing_eligible].copy()

In [22]:
# Retained for audit; these entries are excluded only from pricing analysis.
excluded_zero_price_entries[[
    "AppID", "Name", "Price", "Genres", "total_reviews"
]].head(10)


,AppID,Name,Price,Genres,total_reviews
20,2713630,Black Market,0.0,"Casual,Indie,Simulation",0
23,2072070,RPG Plus - Virtual Tabletop,0.0,"RPG,Strategy",0
39,2731180,Blaster Multiplayer,0.0,Action,0
40,1186670,Fateless,0.0,Action,35
41,830470,MazeBot,0.0,"Action,Casual,Indie",43
46,2641000,Spider-Guy: Trapped in the Cheese Place,0.0,"Action,Adventure,Indie",0
96,1952770,ConquestAreas,0.0,"Action,Casual,Massively Multiplayer,Strategy",0
169,2073030,Priestess Lust,0.0,Action,0
220,1770180,Heroes of Agora,0.0,Action,0
272,671700,Dead Horizon: Origin,0.0,"Action,Adventure,Indie",373


### Population counts after all exclusions

In [23]:
population_audit = pd.Series({
    "Source entries": len(df),
    "Product-filtered entries": len(analysis_df),
    "Entries after historical cutoff": int(after_review_cutoff.sum()),
    "Unparsed release dates": int(analysis_df["release_date"].isna().sum()),
    "Historical entries before title exclusions": len(historical_candidates),
    "Playtest/demo title matches excluded": int(test_or_demo.sum()),
    "Historical entries before zero-price rule": len(historical_before_price_rule),
    "Zero-price entries without Free To Play label excluded": len(excluded_zero_price_entries),
    "Excluded zero-price entries with 100+ reviews": int(excluded_zero_price_entries["total_reviews"].ge(100).sum()),
    "Zero-price entries retained with Free To Play label": int(historical_df["Price"].eq(0).sum()),
    "Historical entries retained": len(historical_df),
    "Historical entries with recorded reviews": int(historical_df["total_reviews"].gt(0).sum()),
    "Historical entries with 100+ reviews": int(historical_df["total_reviews"].ge(100).sum()),
})
population_audit

Source entries                                            125855
Product-filtered entries                                  124129
Entries after historical cutoff                             3479
Unparsed release dates                                         0
Historical entries before title exclusions                120650
Playtest/demo title matches excluded                        8187
Historical entries before zero-price rule                 112463
Zero-price entries without Free To Play label excluded      5619
Excluded zero-price entries with 100+ reviews                896
Zero-price entries retained with Free To Play label        11819
Historical entries retained                               106844
Historical entries with recorded reviews                   78131
Historical entries with 100+ reviews                       21641
dtype: int64

### 3.4 Understand recorded prices: source audit — October 23, 2026

The [September 2024 scraper](https://github.com/FronkonGames/Steam-Games-Scraper/blob/6f0398a0d3037eb1e68f2347ebc5a5866e24ca26/SteamGamesScraper.py) and the [current code](https://github.com/FronkonGames/Steam-Games-Scraper/blob/main/SteamGamesScraper.py) skip previously stored AppIDs during normal collection. File updates therefore do not establish that existing records were refreshed. Price comes from Steam's final-price field; discount and recorded review counts come from SteamSpy, with zero fallbacks when supplemental data is unavailable. The exact version and run used for this file remain unverified.

The snapshot below checks all 10 qualifying entries above $60 and 8 diagnostic lower-price entries against Steam's live US API. Nine live final prices match, six differ, and three have no live price overview. These counts are not a dataset accuracy estimate. The lower-price sample spans four price groups with discounted/undiscounted examples, selected reproducibly with random_state=42.

**Historical prices remain unverified because collection dates are missing.** Today's match does not validate an unknown historical observation; today's difference does not prove an old value wrong. No source price or review value is overwritten. Estimated regular prices are conditional calculations using fields from separate sources, not validated historical base prices.

The audit observations are embedded so the notebook can run offline; this cell does not make fresh API requests. The source URLs identify the live endpoints used. SteamDB corroborates historical pricing examples but cannot connect them to the missing dataset observation dates. Current and historical prices must not be mixed into a supposedly synchronous snapshot.

In [24]:
price_validation_snapshot = pd.DataFrame([{'AppID': 904960, 'Title': 'Cine Tracer', 'Recorded USD': 89.99, 'Recorded discount (%)': 0.0, 'Live final USD': 89.99, 'Live initial USD': 89.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=904960&cc=us&l=english'}, {'AppID': 2537590, 'Title': 'Microsoft Flight Simulator 2024', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': 69.99, 'Live initial USD': 69.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2537590&cc=us&l=english'}, {'AppID': 1250650, 'Title': 'AI＊Shoujo/AI＊少女', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': 69.99, 'Live initial USD': 69.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=1250650&cc=us&l=english'}, {'AppID': 2488620, 'Title': 'F1® 24', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': None, 'Live initial USD': None, 'Live discount (%)': None, 'Comparison': 'No current price returned', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2488620&cc=us&l=english'}, {'AppID': 2669320, 'Title': 'EA SPORTS FC™ 25', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': None, 'Live initial USD': None, 'Live discount (%)': None, 'Comparison': 'No current price returned', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2669320&cc=us&l=english'}, {'AppID': 2069310, 'Title': 'RealFlight Evolution', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': 69.99, 'Live initial USD': 69.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2069310&cc=us&l=english'}, {'AppID': 2878980, 'Title': 'NBA 2K25', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': 69.99, 'Live initial USD': 69.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2878980&cc=us&l=english'}, {'AppID': 1881420, 'Title': 'Cardfight!! Vanguard Dear Days', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': 4.89, 'Live initial USD': 4.89, 'Live discount (%)': 0.0, 'Comparison': 'Differs from live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=1881420&cc=us&l=english'}, {'AppID': 2457540, 'Title': 'Cardfight!! Vanguard Dear Days 2', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': 69.99, 'Live initial USD': 69.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2457540&cc=us&l=english'}, {'AppID': 2582560, 'Title': 'EA SPORTS™ Madden NFL 25', 'Recorded USD': 69.99, 'Recorded discount (%)': 0.0, 'Live final USD': None, 'Live initial USD': None, 'Live discount (%)': None, 'Comparison': 'No current price returned', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2582560&cc=us&l=english'}, {'AppID': 777770, 'Title': 'WAR PARTY', 'Recorded USD': 3.49, 'Recorded discount (%)': 0.0, 'Live final USD': 9.99, 'Live initial USD': 9.99, 'Live discount (%)': 0.0, 'Comparison': 'Differs from live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=777770&cc=us&l=english'}, {'AppID': 492500, 'Title': 'Heroes of the Monkey Tavern', 'Recorded USD': 4.99, 'Recorded discount (%)': 50.0, 'Live final USD': 7.99, 'Live initial USD': 7.99, 'Live discount (%)': 0.0, 'Comparison': 'Differs from live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=492500&cc=us&l=english'}, {'AppID': 1271700, 'Title': 'HOT WHEELS UNLEASHED™', 'Recorded USD': 5.99, 'Recorded discount (%)': 0.0, 'Live final USD': 29.99, 'Live initial USD': 29.99, 'Live discount (%)': 0.0, 'Comparison': 'Differs from live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=1271700&cc=us&l=english'}, {'AppID': 2139300, 'Title': 'Redemption Reapers', 'Recorded USD': 9.99, 'Recorded discount (%)': 60.0, 'Live final USD': 24.99, 'Live initial USD': 24.99, 'Live discount (%)': 0.0, 'Comparison': 'Differs from live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=2139300&cc=us&l=english'}, {'AppID': 3080880, 'Title': 'XiuzhenWorld2', 'Recorded USD': 12.99, 'Recorded discount (%)': 0.0, 'Live final USD': 12.99, 'Live initial USD': 12.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=3080880&cc=us&l=english'}, {'AppID': 1066890, 'Title': 'Automobilista 2', 'Recorded USD': 11.99, 'Recorded discount (%)': 70.0, 'Live final USD': 11.99, 'Live initial USD': 39.99, 'Live discount (%)': 70.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=1066890&cc=us&l=english'}, {'AppID': 298050, 'Title': 'Master of Orion', 'Recorded USD': 24.99, 'Recorded discount (%)': 0.0, 'Live final USD': 24.99, 'Live initial USD': 24.99, 'Live discount (%)': 0.0, 'Comparison': 'Matches live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=298050&cc=us&l=english'}, {'AppID': 1875830, 'Title': 'Shin Megami Tensei V: Vengeance', 'Recorded USD': 23.99, 'Recorded discount (%)': 60.0, 'Live final USD': 59.99, 'Live initial USD': 59.99, 'Live discount (%)': 0.0, 'Comparison': 'Differs from live final price', 'Source URL': 'https://store.steampowered.com/api/appdetails?appids=1875830&cc=us&l=english'}])

price_validation_snapshot

,AppID,Title,Recorded USD,Recorded discount (%),Live final USD,Live initial USD,Live discount (%),Comparison,Source URL
0,904960,Cine Tracer,89.99,0.0,89.99,89.99,0.0,Matches live final price,https://store.steampowered.com/api/appdetails?...
1,2537590,Microsoft Flight Simulator 2024,69.99,0.0,69.99,69.99,0.0,Matches live final price,https://store.steampowered.com/api/appdetails?...
2,1250650,AI＊Shoujo/AI＊少女,69.99,0.0,69.99,69.99,0.0,Matches live final price,https://store.steampowered.com/api/appdetails?...
3,2488620,F1® 24,69.99,0.0,NaN,NaN,NaN,No current price returned,https://store.steampowered.com/api/appdetails?...
4,2669320,EA SPORTS FC™ 25,69.99,0.0,NaN,NaN,NaN,No current price returned,https://store.steampowered.com/api/appdetails?...
5,2069310,RealFlight Evolution,69.99,0.0,69.99,69.99,0.0,Matches live final price,https://store.steampowered.com/api/appdetails?...
6,2878980,NBA 2K25,69.99,0.0,69.99,69.99,0.0,Matches live final price,https://store.steampowered.com/api/appdetails?...
7,1881420,Cardfight!! Vanguard Dear Days,69.99,0.0,4.89,4.89,0.0,Differs from live final price,https://store.steampowered.com/api/appdetails?...
8,2457540,Cardfight!! Vanguard Dear Days 2,69.99,0.0,69.99,69.99,0.0,Matches live final price,https://store.steampowered.com/api/appdetails?...
9,2582560,EA SPORTS™ Madden NFL 25,69.99,0.0,NaN,NaN,NaN,No current price returned,https://store.steampowered.com/api/appdetails?...


### 3.5 Inspect recorded prices and define groups

Inspect the distribution, highest recorded prices, and discount coverage before assigning the chosen price intervals. The zero-price category explicitly requires the Free To Play label.

### Conditional price estimate

Reverse the recorded discount only where the fields permit it. This assumes the Steam price and SteamSpy discount refer to the same offer; the estimate is not a verified regular or launch price. The main analysis uses recorded prices.


In [25]:
discount_fraction = historical_df["Discount"] / 100
valid_regular_price = (
    historical_df["Price"].gt(0)
    & historical_df["Discount"].lt(100)
)
historical_df["estimated_regular_price"] = historical_df["Price"].div(
    (1 - discount_fraction).where(valid_regular_price)
)

## 4. Indie analysis population — 2019–2025

**Working scope for subsequent Indie analyses:** exact Indie-labelled entries with recorded release dates from January 1, 2019 through December 25, 2025, inclusive. Start from the historical population after software/video and explicit Playtest/Demo exclusions and the zero-price eligibility rule. Prologues remain included. Recorded Indie status does not independently establish studio size or ownership.

Create `indie_focus_df` without imposing a review threshold; `historical_df` is the intermediate population before the Indie/date restriction. Create `indie_focus_reviewed_df` with at least 100 recorded reviews for reception comparisons. Low/zero recorded review counts are retained in the broader scope but may represent incomplete collection; they cannot automatically be interpreted as commercial failure or lack of audience.

Revised price intervals: zero + Free To Play label; (0,5]; (5,10]; (10,15]; (15,20]; (20,30]; above 30 USD. Upper boundaries are included. Use the same intervals for conditional estimated regular prices, leaving unreconstructable prices unclassified. These are recorded prices, potentially discounted or stale, not launch prices. The analysis notebooks use these final boundaries.

In [26]:
indie_start_date = pd.Timestamp("2019-01-01")
indie_end_date = pd.Timestamp("2025-12-25")

focus_indie_label = historical_df["Genres"].fillna("").str.split(",").apply(
    lambda labels: "Indie" in {label.strip() for label in labels}
)
indie_focus_df = historical_df.loc[
    focus_indie_label
    & historical_df["release_date"].between(indie_start_date, indie_end_date)
].copy()

indie_price_bins = [-np.inf, 0, 5, 10, 15, 20, 30, np.inf]
indie_price_labels = [
    "0 (F2P label)", "(0, 5]", "(5, 10]", "(10, 15]",
    "(15, 20]", "(20, 30]", "Above 30",
]

# Assign the final price groups in the Indie population.
indie_focus_df["price_group"] = pd.cut(
    indie_focus_df["Price"], bins=indie_price_bins,
    labels=indie_price_labels, right=True,
)
indie_focus_df["regular_price_group"] = pd.cut(
    indie_focus_df["estimated_regular_price"].round(2),
    bins=indie_price_bins, labels=indie_price_labels, right=True,
)

indie_focus_reviewed_df = indie_focus_df.loc[
    indie_focus_df["total_reviews"].ge(100)
].copy()

indie_focus_audit = pd.Series({
    "Eligible Indie-labelled entries (all review counts)": len(indie_focus_df),
    "Entries with zero recorded reviews": int(indie_focus_df["total_reviews"].eq(0).sum()),
    "Entries with at least one recorded review": int(indie_focus_df["total_reviews"].gt(0).sum()),
    "Entries with 100+ recorded reviews": len(indie_focus_reviewed_df),
    "Zero-price / Free To Play-labelled entries": int(indie_focus_df["Price"].eq(0).sum()),
    "Positive recorded-price entries": int(indie_focus_df["Price"].gt(0).sum()),
})
indie_focus_audit

Eligible Indie-labelled entries (all review counts)    61426
Entries with zero recorded reviews                     19538
Entries with at least one recorded review              41888
Entries with 100+ recorded reviews                      8467
Zero-price / Free To Play-labelled entries              6987
Positive recorded-price entries                        54439
dtype: int64

### Revised recorded-price group sizes

Counts cover the broader refined Indie population, with the 100-review subset shown separately. The zero-price reference is kept separate from all paid-price groups. All counts are recorded observations; coverage percentages should not be interpreted as probabilities of commercial success.

In [27]:
indie_focus_price_coverage = indie_focus_df.groupby(
    "price_group", observed=False
).agg(
    total_entries=("AppID", "size"),
    with_recorded_reviews=("total_reviews", lambda values: values.gt(0).sum()),
    with_100_plus_reviews=("total_reviews", lambda values: values.ge(100).sum()),
)
indie_focus_price_coverage["pct_with_100_plus"] = (
    indie_focus_price_coverage["with_100_plus_reviews"]
    / indie_focus_price_coverage["total_entries"].replace(0, np.nan) * 100
)
indie_focus_price_coverage.round(1)

,total_entries,with_recorded_reviews,with_100_plus_reviews,pct_with_100_plus
price_group,,,,
0 (F2P label),6987,2284,787,11.3
"(0, 5]",38003,27988,4040,10.6
"(5, 10]",10777,7663,2130,19.8
"(10, 15]",3474,2474,958,27.6
"(15, 20]",1305,936,375,28.7
"(20, 30]",434,300,144,33.2
Above 30,446,243,33,7.4


## 5. Classify supported play modes

Use only the final Indie sample with 100+ recorded reviews. Give co-op precedence over other multiplayer labels, then classify single-player-only entries. Preserve unknown modes separately.


In [28]:
mode_df = indie_focus_reviewed_df.copy()

category_sets = (
    mode_df["Categories"]
    .fillna("")
    .str.split(",")
    .apply(lambda labels: {label.strip().casefold() for label in labels if label.strip()})
)

coop_labels = {
    "co-op", "online co-op", "lan co-op", "shared/split screen co-op",
}
multiplayer_labels = {
    "multi-player", "multiplayer", "pvp", "online pvp", "lan pvp",
    "shared/split screen pvp", "shared/split screen",
    "cross-platform multiplayer", "mmo",
}

mode_df["supports_single_player"] = category_sets.apply(
    lambda labels: "single-player" in labels
)
mode_df["supports_coop"] = category_sets.apply(
    lambda labels: bool(labels & coop_labels)
)
mode_df["supports_multiplayer"] = (
    mode_df["supports_coop"]
    | category_sets.apply(lambda labels: bool(labels & multiplayer_labels))
)
mode_df["remote_together_only_signal"] = (
    category_sets.apply(lambda labels: "remote play together" in labels)
    & ~mode_df["supports_multiplayer"]
)

mode_order = [
    "Single-player only (recorded)",
    "Multiplayer without recorded co-op",
    "Co-op-capable",
    "Unknown/unclassified",
]

mode_df["play_mode"] = np.select(
    [
        mode_df["supports_coop"],
        mode_df["supports_multiplayer"],
        mode_df["supports_single_player"] & ~mode_df["remote_together_only_signal"],
    ],
    [mode_order[2], mode_order[1], mode_order[0]],
    default=mode_order[3],
)
mode_df["play_mode"] = pd.Categorical(
    mode_df["play_mode"], categories=mode_order, ordered=True
)

mode_df.groupby("play_mode", observed=False).size().rename("Entries")

play_mode
Single-player only (recorded)         6491
Multiplayer without recorded co-op     509
Co-op-capable                         1387
Unknown/unclassified                    80
Name: Entries, dtype: int64

## 6. Limitations

- Coverage is incomplete and review freshness is uneven. The source update date and recorded release dates are not per-record collection timestamps. The historical cutoff cannot guarantee completeness before that boundary.
- Zero review counts may reflect no reviews or unavailable/incomplete collection. Reception is undefined for those entries; no review values are imputed.
- Zero recorded prices may mix genuinely free titles and unavailable prices. Positive prices may be discounted or stale. Estimated regular prices depend on the recorded discount and do not recover launch prices.
- Software/video exclusions use recorded labels and may remove hybrid products. An English title rule excludes explicit Playtest/Demo matches, but some non-game entries or unrecognized demos/playtests can remain. Prologues are retained. Missing genres do not establish that an entry belongs to any particular genre.
- Genre groups overlap. Some labels describe business models, status, or content rather than genre. The pricing population retains zero prices only with the Free To Play label; positive-price entries with that label remain under other genres but are excluded from the Free To Play row.
- Review thresholds select different populations; results do not represent the entire Steam catalog. Sparse groups are descriptive examples, not reliable market-wide estimates.
- Associations do not establish that changing price causes reception to change. Release period, exposure, genre, publisher, and other factors can affect both.